# Relational databases and first queries

After this block you can explain what tables, rows, columns, data types and keys are, start a PostgreSQL database server on your own computer and connect to it from Python, create tables with constraints and load data into them with `COPY`, and write queries that filter, sort and limit rows, including rows with missing values. The block serves the first two learning outcomes of Session 3: *explain tables, keys and relationships in a relational database* and *create tables in PostgreSQL, load data and write SQL queries that filter and sort*.

The running example is the Berlin snapshot of Inside Airbnb (26 June 2026, CC BY 4.0) in the course database `airbnb`, with three tables: `listings` (one row per listing), `calendar` (one row per listing and night of the coming year) and `reviews` (number of reviews per listing and month). No prior knowledge of databases is assumed. The course uses **PostgreSQL**, one of the most widely used SQL databases; its SQL dialect is close to the SQL standard, so what you learn here transfers to other databases with few changes. A Python package starts PostgreSQL on your own computer, so there is nothing to install by hand.

*Author: course team · Licence: CC BY 4.0, except the sections marked "Adapted from", which keep their source licence (named there).*

## Overview

| Section | What it covers |
|---|---|
| Setup | starting a local PostgreSQL server from Python, building the course database, the helper `q`, a first look at the tables and columns |
| What a relational database is: tables, rows, columns and data types | tables, rows and columns, databases against CSV files, PostgreSQL's data types, real dates and date arithmetic, strict types, `CAST` and `::` |
| Primary keys, foreign keys and relationships | primary and foreign keys, one-to-many relationships, normalisation, joins on paper, checking the keys of the course data, adding a foreign key to data that break it, star schemas |
| PostgreSQL: a database server on your computer | servers and clients, connection strings, three ways to connect, SQL dialects, `information_schema` and `pg_catalog`, read-only connections |
| Creating tables and loading data: CREATE TABLE, INSERT, COPY | constraints, `INSERT ... RETURNING`, `UPDATE` and `DELETE`, `executemany`, transactions, `COPY`, `to_sql`, a strict schema on the real listings |
| Queries: SELECT, WHERE, ORDER BY, LIMIT | the clause template and its logical order, a `SELECT` alias in `WHERE` and in `ORDER BY`, `SELECT` with `DISTINCT`, `WHERE` with `AND`/`OR`, `IN`, `BETWEEN`, text patterns with `LIKE`, `ILIKE`, `ESCAPE` and regular expressions, dates, `ORDER BY`, `LIMIT` and `OFFSET`, `CASE` |
| NULL values | three-valued logic, `IS NULL`, `COUNT(*)` against `COUNT(col)`, `COALESCE`, `IS DISTINCT FROM` and the sort order, whether missing prices are random |
| Worked example | a weekend flat for a family, checked against its calendar |
| Check your understanding, Exercises, Debrief | five questions, ten exercises in pairs, what to compare afterwards |

Where SQL sits in an analysis: a question becomes a query, the database server returns a small result, and Python takes it from there.

```mermaid
flowchart LR
    Q["Business question"] --> S["SQL query"]
    S --> DB[("PostgreSQL server<br/>database airbnb:<br/>listings, calendar,<br/>reviews")]
    DB --> R["Small result table"]
    R --> P["pandas<br/>in Python"]
    P --> A["Chart, model, report"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Setup: open the course database

### Starting a local PostgreSQL server

PostgreSQL is a **database server**: a separate program that holds the data and waits for queries from other programs, its **clients**. In a company, the IT department runs it on a machine of its own. The package `pixeltable-pgserver` ships the PostgreSQL programs inside a Python package, so `pgserver.get_server(folder)` can start a private server on your own computer. The server is **local**: only programs on the same computer can reach it. On macOS and Linux it opens no network port at all and uses a **socket file** (a file through which programs on one computer talk to each other); on Windows it listens on the address `127.0.0.1`, which always means "this computer", on a free port. The server keeps its data in the folder you name, here `case-study/data/airbnb/pgdata/`; the first start creates the folder.

The server **keeps running in the background**: it is a second program on your computer, and it must run for as long as the notebook sends queries. It stops by itself when the last Python program that uses it ends, for example when you shut down or restart the notebook's kernel (`cleanup_mode="stop"`, the default). The data stay in the folder, and the next start takes a few seconds.

> **Note.** If `import pixeltable_pgserver` fails, run `uv sync` in the repository folder and restart the kernel.

In [2]:
import subprocess
import sys

import pandas as pd
import pixeltable_pgserver as pgserver
import psycopg

PGDATA = Path("case-study/data/airbnb/pgdata")         # the folder where the server keeps its data
# Start the server, or reuse it if it already runs. It runs locally, on this computer only:
# on macOS and Linux it opens no network port and listens on a socket file in a temporary folder;
# on Windows it listens on 127.0.0.1 (this computer) on a free port that it picks itself.
# server.get_uri() returns the address, so you never type a host or port yourself.
server = pgserver.get_server(PGDATA)

### Building the course database

The build script [`make_airbnb_postgres.py`](make_airbnb_postgres.py) creates the database `airbnb` on this server and loads the three Parquet files with `COPY` (the section on loading data explains it). It needs the Airbnb files in `case-study/data/airbnb/`, downloaded as `airbnb-v1.zip` from the course drive (see [case-study/README.md](../../../case-study/README.md)). The first build takes a few seconds; afterwards the script sees that the tables exist and stops.

`subprocess.run([sys.executable, "script.py"], check=True)` starts the build script as a second Python program on your computer; `sys.executable` is the Python that runs this notebook. The notebook waits until the script has finished. The script connects to the server that the previous cell started; it does not start a second one. `check=True` raises an error if the script fails.

In [3]:
# runs the script locally and waits until it ends; the script itself leaves nothing running
build_run = subprocess.run([sys.executable, "sessions/03-sql/workbooks/make_airbnb_postgres.py"], check=True)

the tables exist; use --force to rebuild them
done in 0 s


### Connecting from Python

`psycopg.connect(uri)` opens a **connection** to the server; `psycopg` is the standard PostgreSQL driver for Python. The **connection string** (URI) names the user, the server and the database; `server.get_uri("airbnb")` builds it for the database `airbnb`, which the build script created with `CREATE DATABASE airbnb`. One server can hold several databases; a new server has only the default database `postgres`. Three settings follow:

- `autocommit=True` makes every statement its own transaction. Without it, the first error in a notebook leaves the connection in a failed transaction, and PostgreSQL refuses every later statement with `current transaction is aborted` until you call `con.rollback()`. The section on transactions shows how to group statements on purpose.
- PostgreSQL returns `NUMERIC` values (exact decimals, used for `price`) as Python `Decimal` objects. `register_loader("numeric", FloatLoader)` turns them into ordinary floats, which pandas handles better.
- The helper `q(sql, params)` runs a query and returns the result as a pandas DataFrame, which is easier to read than a list of tuples. A **cursor** (`con.cursor()`) runs one statement and holds its result; `cur.description` holds the names of the result columns. Block 2 explains cursors and the `params` argument.

In [4]:
from psycopg.types.numeric import FloatLoader

# "airbnb" is the name of the database; make_airbnb_postgres.py created it (DATABASE = "airbnb").
# One server can hold several databases; a new server has only the default database "postgres".
con = psycopg.connect(server.get_uri("airbnb"), autocommit=True)
con.adapters.register_loader("numeric", FloatLoader)                # NUMERIC values arrive as floats


def q(sql, params=None):
    """Run an SQL query on the course database and return the result as a pandas DataFrame."""
    with con.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[column.name for column in cur.description])


q("SELECT version()")

,version
0,PostgreSQL 16.15 on aarch64-apple-darwin24.6.0...


### Stopping the server

You do not need to stop the server: it stops when the kernel shuts down. To stop it earlier, run `con.close()` and then `server.cleanup()`; the Setup cells start it again. To delete the course database completely, stop the server and delete the folder `case-study/data/airbnb/pgdata/`; the build script recreates it from the Parquet files.

### A first look at the tables and columns

Before you query a database, find out which tables it has and which columns each table has. Every PostgreSQL database describes itself in the schema `information_schema`, which is part of the SQL standard (MySQL and Microsoft SQL Server have it too). `information_schema.tables` lists the tables. A **schema** in PostgreSQL is a named group of tables inside a database, like a folder; the course tables lie in the schema `public`, the default place for tables.

**Predict.** How many tables does the course database have?

In [5]:
q("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' ORDER BY table_name")

,table_name
0,calendar
1,listings
2,reviews


`information_schema.columns` has one row per column of every table. `ordinal_position` is the position of the column in the table, and `data_type` its declared type.

In [6]:
# find all the columns of the table listings
q("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'public' AND table_name = 'listings'
ORDER BY ordinal_position
""")

,column_name,data_type
0,id,bigint
1,name,text
2,host_id,bigint
3,district,text
4,neighbourhood,text
5,latitude,double precision
6,longitude,double precision
7,room_type,text
8,accommodates,integer
9,bedrooms,integer


The same query with `'calendar'` or `'reviews'` lists the columns of the other two tables. Note the types `date` and `boolean`: the next section explains them.

In [7]:
q("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'public' AND table_name = 'calendar'
ORDER BY ordinal_position
""")

,column_name,data_type
0,listing_id,bigint
1,date,date
2,available,boolean


## What a relational database is: tables, rows, columns and data types

### Tables, rows and columns

A **relational database** stores data in **tables**. Each **row** is one record, for example one listing; each **column** has a name and a **data type** (whole number, decimal number, text, date). The set of table definitions is the **schema**. The idea goes back to Edgar F. Codd (1970), who proposed that data should be described by tables and queried by their content, not by the way they are stored on disk. A program that stores such tables and answers queries on them is a **database management system** (DBMS); PostgreSQL, MySQL, Microsoft SQL Server, Oracle and SQLite are examples.

You will meet several words for the same things. A table is also called a **relation**, a row a **tuple** or **record**, and a column an **attribute** or **field**. A table looks much like a pandas DataFrame, with two differences: its rows have no built-in order, and its columns have declared types and rules that the database checks on every write.

Why not just keep CSV files? Files work well for small data that one person analyses. A database adds reliable storage that survives crashes, fast access to data that does not fit into memory, rules that block invalid values (a negative price, a review for a listing that does not exist), control over who may read and write, and safe concurrent use by several programs.

A small example that you can follow by hand (the ids are invented; the real ones have up to 19 digits):

| listings |  |  |  |
|---|---|---|---|
| **id** | district | room_type | price |
| 1 | Mitte | Entire home/apt | 120 |
| 2 | Neukölln | Private room | 45 |
| 3 | Mitte | Entire home/apt | NULL |

| reviews |  |  |
|---|---|---|
| **listing_id** | **month** | n_reviews |
| 1 | 2026-04-01 | 3 |
| 1 | 2026-05-01 | 2 |
| 2 | 2026-05-01 | 1 |

`NULL` marks a value that is missing or unknown: listing 3 shows no price. The section on NULL values treats it in full.

*Terminology and the comparison with CSV files adapted from [Data 100, SQL I](https://github.com/DS-100/fa23-course-notes/blob/main/sql_I/sql_I.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

Let us look at the real `listings` table. `SELECT columns FROM table ORDER BY id LIMIT 3` returns the named columns of the first three rows by id (the section on queries explains the clauses). We leave out the listing title on purpose: hosts sometimes write their own name into it, and an analysis does not need it.

In [8]:
q("SELECT id, host_id, district, room_type, accommodates, price, minimum_nights, first_review FROM listings ORDER BY id LIMIT 3")

,id,host_id,district,room_type,accommodates,price,minimum_nights,first_review
0,3176,3718,Pankow,Entire home/apt,2,160.71,2,2009-06-20
1,9991,33852,Pankow,Entire home/apt,7,193.33,6,2015-08-09
2,14325,55531,Pankow,Entire home/apt,1,NaN,180,2010-06-29


### Data types in PostgreSQL

When you create a table, you declare each column with a type, and PostgreSQL checks every value against it. The types of the course database:

| Type | Holds | Write a value as | Example in the course data |
|---|---|---|---|
| `INTEGER`, `BIGINT` | a whole number (4 or 8 bytes) | `42` | `accommodates`; `id` needs `BIGINT`, because some ids have 19 digits |
| `NUMERIC(10, 2)` | an exact decimal number, here with 10 digits, 2 after the point | `160.71` | `price` |
| `DOUBLE PRECISION` | a floating-point number (8 bytes) | `52.5` | `latitude`, `longitude` |
| `TEXT`, `VARCHAR(50)` | text of any length; text of at most 50 characters | `'Mitte'` | `district`, `room_type` |
| `BOOLEAN` | true or false | `TRUE`, `FALSE` | `available` in `calendar` |
| `DATE` | a day | `DATE '2026-07-01'` | `first_review`, the night in `calendar` |
| `TIMESTAMP`, `INTERVAL` | a day and a time; a length of time | `TIMESTAMP '2026-07-01 14:30'`, `INTERVAL '3 days'` | (not stored) |

Every type also accepts `NULL`, a missing value. `NUMERIC` is exact: 0.1 + 0.2 is exactly 0.3, which matters for money. `DOUBLE PRECISION` is the same floating-point type as a Python `float`: fast, but with tiny rounding errors.

`pg_typeof(x)` returns the type of a value. **Predict.** What type does the comparison `price > 100` have?

In [9]:
q("""
SELECT pg_typeof(id) AS id_type, pg_typeof(price) AS price_type, pg_typeof(first_review) AS first_review_type,
       pg_typeof(price > 100) AS comparison_type, price > 100 AS is_expensive
FROM listings
WHERE price IS NOT NULL
LIMIT 1
""")

,id_type,price_type,first_review_type,comparison_type,is_expensive
0,bigint,numeric,date,boolean,True


A comparison is a `boolean`, and pandas shows it as `True` or `False`.

### Dates are real dates

Because `first_review` is a `date`, `MIN` and `MAX` return the earliest and latest day, and date arithmetic works directly. `DATE '2026-06-26'` writes a date value (a **typed literal**), and the difference of two dates is a whole number of days.

In [10]:
q("""
SELECT MIN(first_review) AS earliest_first_review,
       MAX(last_review)  AS latest_last_review,
       DATE '2026-06-26' - MIN(first_review) AS days_since_earliest      -- integer days
FROM listings
""")

,earliest_first_review,latest_last_review,days_since_earliest
0,2009-06-20,2026-07-02,6215


More date functions:

- `EXTRACT(YEAR FROM d)` returns a part of a date: `YEAR`, `MONTH`, `DAY`, `DOW` (day of the week, 0 = Sunday) or `ISODOW` (1 = Monday to 7 = Sunday);
- `date_trunc('month', d)` cuts a date back to the first day of its month (the result is a timestamp; `::date` makes it a date again);
- `d + INTERVAL '1 year'` adds a length of time, and `AGE(a, b)` returns the difference as years, months and days.

`::int` and `::text` convert the results for display; the subsection on `CAST` below explains them.

In [11]:
q("""
SELECT id, first_review,
       EXTRACT(YEAR FROM first_review)::int     AS first_year,
       date_trunc('month', first_review)::date  AS first_month,
       first_review + INTERVAL '1 year'          AS one_year_later,
       AGE(DATE '2026-06-26', first_review)::text AS time_on_airbnb
FROM listings
WHERE first_review IS NOT NULL
ORDER BY first_review
LIMIT 3
""")

,id,first_review,first_year,first_month,one_year_later,time_on_airbnb
0,3176,2009-06-20,2009,2009-06-01,2010-06-20,17 years 6 days
1,17904,2010-02-18,2010,2010-02-01,2011-02-18,16 years 4 mons 8 days
2,23834,2010-06-07,2010,2010-06-01,2011-06-07,16 years 19 days


**Predict.** What does PostgreSQL do with `DATE '2009-13-01'`, a date in a thirteenth month? The next cell is meant to fail: `try`/`except` catches the error, so that the notebook keeps running.

In [12]:
try:
    q("SELECT DATE '2009-13-01' AS thirteenth_month")
except psycopg.Error as error:
    print(type(error).__name__, "|", error)

DatetimeFieldOverflow | date/time field value out of range: "2009-13-01"
LINE 1: SELECT DATE '2009-13-01' AS thirteenth_month
                    ^
HINT:  Perhaps you need a different "datestyle" setting.


Read the message from the top. `DatetimeFieldOverflow` is the kind of error; `date/time field value out of range: "2009-13-01"` names the value; `LINE 1:` and the `^` point to the place in the query. A real date type rejects an impossible date when it is written, so an impossible date can never enter a `DATE` column.

> **Note.** Databases differ here. SQLite, a small database stored in one file that is built into phones and many apps, has no date type: it stores dates as text such as `'2026-06-26'` and would store `'2009-13-01'` without complaint.

### Strict types

**Predict.** A column is declared `INTEGER`. Which of the values `3`, `'4'` and `'three'` does PostgreSQL accept? `CREATE TEMP TABLE` creates a table that disappears when the connection closes.

In [13]:
con.execute("CREATE TEMP TABLE stay (nights INTEGER)")
for value in ["3", "'4'", "'three'"]:
    try:
        con.execute(f"INSERT INTO stay VALUES ({value})")
        print(f"{value:8} accepted")
    except psycopg.Error as error:
        print(f"{value:8} rejected: {error.diag.message_primary}")
print(con.execute("SELECT nights, pg_typeof(nights) FROM stay").fetchall())

3        accepted
'4'      accepted
'three'  rejected: invalid input syntax for type integer: "three"
[(3, 'integer'), (4, 'integer')]


Text that is a valid integer, `'4'`, is converted to the number 4; `'three'` is rejected. A column of numbers can therefore never contain a word that a later `AVG` would have to deal with. `error.diag.message_primary` is the first line of the error message, without the `LINE` and `^` part.

### Converting types with `CAST` and `::`

`CAST(expression AS type)` converts a value to another type, for example a number stored as text to an integer, and `AS name` gives the result column a readable name. Without `AS`, the column is named after the function or the type. PostgreSQL also accepts the short form `expression::type`. **Predict.** Is `CAST(160.71 AS INTEGER)` 160 or 161?

In [14]:
q("""
SELECT price,
       CAST(price AS INTEGER)       AS price_cast,
       price::integer               AS price_short_form,
       TRUNC(price)::integer        AS price_cut_off,
       '12'::integer + 1            AS text_to_number,
       accommodates::text || ' guests' AS guests_text      -- || joins text
FROM listings
WHERE price IS NOT NULL
ORDER BY id
LIMIT 3
""")

,price,price_cast,price_short_form,price_cut_off,text_to_number,guests_text
0,160.71,161,161,160,13,2 guests
1,193.33,193,193,193,13,7 guests
2,10.88,11,11,10,13,2 guests


PostgreSQL **rounds** when it converts a decimal to an integer: 160.71 becomes 161. `TRUNC(x)` cuts off the decimals. Other databases differ (SQLite, for example, cuts off), so check a conversion when you move a query to another database.

*`CAST` and column aliases adapted from [Data 100, SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (UC Berkeley Data 100, BSD 3-Clause), here on the Airbnb data.*

### Relational tables in practice

- **Stack Exchange Data Explorer** lets anyone query the public Stack Overflow database with SQL; questions, answers, users and votes are separate tables.
- **Inside Airbnb** publishes each city as separate files (listings, calendar, reviews); the course database turns them into three tables.
- A table with named, typed columns is a contract: every row has the same fields, and every program that reads the table can rely on that. One cell of text in a spreadsheet column of prices is enough to break a calculation.

## Primary keys, foreign keys and relationships

### Primary keys and foreign keys

A **primary key** is a column, or a combination of columns, whose value identifies each row uniquely and is never empty. In the case study, `id` identifies a listing. The review table has a **composite key**: the pair (`listing_id`, `month`) identifies a row, because each listing has at most one row per month. The calendar has the key (`listing_id`, `date`).

A **foreign key** is a column that refers to the primary key of another table. `calendar.listing_id` and `reviews.listing_id` point to `listings.id`. This expresses a **relationship**: one listing has many calendar nights and many review months, each of which belongs to exactly one listing (a *one-to-many* relationship). Other relationships are *one-to-one* (a listing and its single registration record) and *many-to-many* (listings and amenities: a listing has many amenities, and an amenity belongs to many listings; this needs a third table, a **junction table**, with one row per pair and a foreign key to each side).

A relationship starts as a design decision. The database checks it only when `CREATE TABLE` declares it: `REFERENCES` rejects a value that does not exist in the other table, and `PRIMARY KEY` or `UNIQUE` rejects a second matching row. An ER diagram enforces nothing, and a join works without any declared key, because a join only compares values. A rule such as "every book has at least one author" cannot be written as a foreign key: it stays in the design. The course database declares the three primary keys but no foreign keys; the subsection on adding a foreign key shows why.

An **entity-relationship (ER) diagram** is the standard picture of a schema: each table is a box with its columns (`PK` marks the primary key, `FK` a foreign key), and each foreign key is a line. The ends of the line, drawn in *crow's-foot notation*, say how many rows match: `||` exactly one, `o|` zero or one, `|{` one or many, `o{` zero or many. Read each end at the box it touches: one listing has zero or many calendar nights; each night belongs to exactly one listing.

```mermaid
erDiagram
    listings ||--o{ calendar : "has nights"
    listings ||--o{ reviews : "has review months"
    listings {
        BIGINT id PK
        BIGINT host_id
        TEXT district
        TEXT room_type
        NUMERIC price
        INTEGER minimum_nights
        TEXT license_status
    }
    calendar {
        BIGINT listing_id PK, FK
        DATE date PK
        BOOLEAN available
    }
    reviews {
        BIGINT listing_id PK, FK
        DATE month PK
        INTEGER n_reviews
    }
```

*ER diagram syntax: [Mermaid, Entity relationship diagrams](https://mermaid.js.org/syntax/entityRelationshipDiagram.html).*

### Normalisation

Giving each fact one home table is called **normalisation**. That table is the **single source of truth**, and every other table refers to it by key. The district of a listing lives in `listings`, not in each of its 365 calendar rows. If a listing moves to another district, one row changes, and no copy can contradict another. The database does not choose the home table: the table design does.

### Following a relationship: joins

To answer "how many reviews did the listings in Mitte receive?", each review row needs the district of its listing: the database pairs the rows of two tables whose keys match. This operation is a **join**; block 2 teaches its syntax. Three kinds matter, shown on the small tables from the start of the notebook:

- an **inner join** returns one row for each pair of matching rows: listing 1 with its two review months and listing 2 with its one, 3 rows; listing 3 has no partner and disappears;
- a **left join** returns the same pairs and, in addition, every row of the left table without a partner, with NULL in the columns of the right table: 4 rows;
- a **cross join** pairs every row of one table with every row of the other, whether the keys match or not: 3 × 3 = 9 rows.

### Why keys matter

Keys make the relationship between tables explicit, and the database uses them to guard data quality at the moment data are written: a review row for an unknown listing, or the same night of a listing twice, is rejected immediately instead of being discovered months later in an analysis. The database also builds an **index** (a sorted lookup structure, like the index at the back of a book) on every primary key, which makes lookups and joins by key fast.

### Checking the keys of the course data

A primary key must be unique. `COUNT(*)` counts rows and `COUNT(DISTINCT column)` counts the different values of a column; if both are equal, no value occurs twice. For the composite key of the calendar, the inner query in parentheses (a **subquery**, block 2) first keeps the distinct pairs, and the outer query counts them.

**Predict.** The calendar covers 365 nights per listing. Roughly how many rows does it have?

In [15]:
q("""
SELECT (SELECT COUNT(*) FROM listings)                                         AS n_listings,
       (SELECT COUNT(DISTINCT id) FROM listings)                               AS n_ids,
       (SELECT COUNT(*) FROM calendar)                                         AS n_calendar_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT listing_id, date FROM calendar)) AS n_calendar_keys,
       (SELECT COUNT(*) FROM reviews)                                          AS n_review_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT listing_id, month FROM reviews)) AS n_review_keys
""").T.rename(columns={0: "value"})

,value
n_listings,10221
n_ids,10221
n_calendar_rows,3759500
n_calendar_keys,3759500
n_review_rows,180656
n_review_keys,180656


All three keys hold: each count of rows equals the count of distinct keys. Here the check only confirms what the database already guarantees, because the build script declared the primary keys, and a declared key rejects every duplicate when it is written.

### Adding a foreign key to data that break it

`ALTER TABLE ... ADD CONSTRAINT name FOREIGN KEY (column) REFERENCES other_table (column)` declares a foreign key on an existing table. PostgreSQL first checks every existing row. **Predict.** Does every `listing_id` in `reviews` refer to an existing listing?

In [16]:
try:
    con.execute("""
        ALTER TABLE reviews
        ADD CONSTRAINT reviews_listing_fk FOREIGN KEY (listing_id) REFERENCES listings (id)
    """)
except psycopg.errors.ForeignKeyViolation as error:
    print(error)

insert or update on table "reviews" violates foreign key constraint "reviews_listing_fk"
DETAIL:  Key (listing_id)=(1025263460868613348) is not present in table "listings".


The message names the constraint, and the `DETAIL` line names the first key that fails: a review for a listing that is not in `listings`. PostgreSQL refuses the constraint, and the table stays as it was. The query below counts all such rows. `WHERE listing_id NOT IN (SELECT id FROM listings)` keeps the rows whose listing is missing. (This form is safe here because `id` is never NULL; block 2 shows why `NOT IN` is dangerous when the list can contain a NULL.) `UNION ALL` stacks the rows of two queries with the same columns.

In [17]:
q("""
SELECT 'calendar' AS child_table, COUNT(*) AS orphan_rows, COUNT(DISTINCT listing_id) AS unknown_listings
FROM calendar WHERE listing_id NOT IN (SELECT id FROM listings)
UNION ALL
SELECT 'reviews', COUNT(*), COUNT(DISTINCT listing_id)
FROM reviews WHERE listing_id NOT IN (SELECT id FROM listings)
ORDER BY child_table
""")

,child_table,orphan_rows,unknown_listings
0,calendar,28835,79
1,reviews,499,57


The foreign key fails for 79 listings in the calendar (28,835 rows, exactly 365 per listing) and 57 listings in the review table. Inside Airbnb scrapes listings, calendars and reviews separately, and the listings were scraped between 26 June and 3 July 2026: a listing that went offline between two scrapes appears in one file but not in the other. A foreign key would have rejected these rows, which is exactly the point: the database makes you decide what to do with them instead of letting a join lose them silently. The course database keeps all rows without a foreign key, so that you can find them with SQL in block 2.

The scraping over several days also shows in the dates of the calendar:

In [18]:
q("SELECT MIN(date) AS first_night, MAX(date) AS last_night, COUNT(DISTINCT date) AS n_dates FROM calendar")

,first_night,last_night,n_dates
0,2026-06-26,2027-07-02,372


The calendar has 372 different dates, although each listing has exactly 365 rows: each calendar starts on the day its listing was scraped. Questions about "the next weeks" therefore mean slightly different weeks for different listings.

> **Note.** A file is not a database. A Parquet or CSV file has no keys and no constraints: nothing stops a duplicated `id` or a calendar row for a listing that is not in the listings file. When you work with files, check the key properties yourself (as above) and repeat the check after each update.

### Star schemas

Analytical databases often organise their tables as a **star schema**. A central **fact table** holds one row per event or measurement (a sale, a booked night, a review month) and refers by foreign keys to several **dimension tables** that describe the parts of the fact in detail (the product, the customer, the date). The Data 100 example is a shop that sells bubble tea: the fact table lists each offering by keys, and dimension tables describe tea varieties and toppings.

![A fact table with dimension tables: tea, toppings and the offerings that combine them](extras/images/multidimensional.png)

Drawing the foreign keys as lines gives the star its name:

![The same tables as a star schema, with the fact table in the centre](extras/images/star.png)

The course database is a small star: `listings` is the dimension that describes each listing, and `calendar` and `reviews` are fact tables whose rows refer to it.

A **snowflake schema** is a star schema with normalised dimensions: a repeated text such as a product category moves to its own table, and the dimension refers to it by a key. Revenue per category then needs two joins instead of one, but renaming a category changes one row instead of one row per product. Both schemas hold the same information. Data warehouses are read far more often than they change, so a star schema, which copies dimension text on purpose, is the usual first choice ([Kimball Group, Snowflake dimensions](https://www.kimballgroup.com/data-warehouse-business-intelligence-resources/kimball-techniques/dimensional-modeling-techniques/snowflake-dimension/)).

*Figures and the star-schema example from [Data 100, SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

### Keys and star schemas in practice

- **Online shops** keep customers, orders and products in separate tables; an order row refers to a customer id and a product id, so that a change of address is made once.
- **Data quality checks** in finance and retail look for records that refer to a deleted customer or a discontinued product, exactly like the calendar rows of vanished listings here.

## PostgreSQL: a database server on your computer

### What a database server is

Most database systems run as a **server**: a separate program that manages the data, and **clients** (your Python code, a graphical tool, a web application) connect to it and send SQL. The server checks who may connect, runs the queries and returns the results; several clients can read and write at the same time. Clients never open the data files themselves.

| | A database server (PostgreSQL) | Files (CSV, Parquet) |
|---|---|---|
| Where the data live | a data folder that only the server reads and writes | files that every program opens itself |
| What runs | the server, plus your program as a client | only your program |
| How you connect | a connection string: user, host or socket, port, database | the file path |
| Rules and keys | checked by the server on every write | none |
| Many users writing at once | yes | no |

The server in this notebook runs on your own computer and accepts connections only from it. A server of an organisation runs the same program on another machine; only the connection string changes, for example `postgresql://anna@db.example.org:5432/airbnb` (user `anna`, host `db.example.org`, the standard PostgreSQL port 5432, database `airbnb`).

> **Note.** Do not copy or edit the files in `pgdata/` by hand while the server runs. To pass the database to someone else, give them the build script and the Parquet files, or an export made with the tool `pg_dump`.

### Three ways to connect

1. **From Python** with `psycopg.connect(uri)`, which returns a **connection**; this course mostly uses this way;
2. in a graphical **database client**, such as [DBeaver](https://dbeaver.io/) or [pgAdmin](https://www.pgadmin.org/) (both free), with the same connection details;
3. with PostgreSQL's command-line client `psql`. Commands that start with a backslash, such as `\dt` (list the tables) and `\d reviews` (describe one table), belong to `psql`, not to SQL. `server.psql(commands)` runs `psql` with the given input; `\c airbnb` connects to the course database first.

In [19]:
print(server.psql("\\c airbnb\n\\dt\n\\d reviews"))

You are now connected to database "airbnb" as user "postgres".
          List of relations
 Schema |   Name   | Type  |  Owner   
--------+----------+-------+----------
 public | calendar | table | postgres
 public | listings | table | postgres
 public | reviews  | table | postgres
(3 rows)

                Table "public.reviews"
   Column   |  Type   | Collation | Nullable | Default 
------------+---------+-----------+----------+---------
 listing_id | bigint  |           | not null | 
 month      | date    |           | not null | 
 n_reviews  | integer |           | not null | 
Indexes:
    "reviews_pkey" PRIMARY KEY, btree (listing_id, month)




The description of `reviews` lists each column with its type, whether it may be NULL, and the index `reviews_pkey` that PostgreSQL built for the primary key.

### PostgreSQL and other SQL dialects

SQL is an international standard (ISO/IEC 9075), but every database adds its own functions and details, its **dialect**. PostgreSQL follows the standard closely and is open source; it grew out of the POSTGRES project at the University of California, Berkeley, which started in 1986 ([PostgreSQL: History](https://www.postgresql.org/docs/current/history.html)). Other common dialects are MySQL, Microsoft SQL Server, Oracle, SQLite and DuckDB. The core of this session (`SELECT`, `WHERE`, `JOIN`, `GROUP BY`, keys and constraints) works in all of them; functions for dates and text differ most. When a query moves to another database, test it there.

### Looking inside the server

Next to `information_schema`, PostgreSQL describes itself in the schema `pg_catalog`, with functions such as `pg_database_size(name)` and `pg_total_relation_size(table)` (a table with its indexes, in bytes). `pg_size_pretty` writes a size in kB or MB.

**Predict.** Which of the three tables needs the most space, and roughly how many times more than `listings`?

In [20]:
q("""
SELECT pg_size_pretty(pg_database_size('airbnb'))        AS database,
       pg_size_pretty(pg_total_relation_size('listings')) AS listings,
       pg_size_pretty(pg_total_relation_size('calendar')) AS calendar,
       pg_size_pretty(pg_total_relation_size('reviews'))  AS reviews
""")

,database,listings,calendar,reviews
0,296 MB,2968 kB,273 MB,13 MB


The calendar, with 3.8 million rows and an index for its primary key, takes most of the space, about ninety times as much as the listings. PostgreSQL stores some extra information with every row, so that many users can read and write at the same time.

### Read-only connections

When a program only reads, make the connection read-only, so that a mistake cannot change the data. `SET default_transaction_read_only = on` does this for one connection. On a shared server, the administrator usually gives analysts a user who may only read (`GRANT SELECT`).

In [21]:
reader = psycopg.connect(server.get_uri("airbnb"), autocommit=True)
reader.execute("SET default_transaction_read_only = on")
try:
    reader.execute("CREATE TABLE should_fail (x INTEGER)")
except psycopg.errors.ReadOnlySqlTransaction as error:
    print("read-only connection refuses to write:", error)
reader.close()

read-only connection refuses to write: cannot execute CREATE TABLE in a read-only transaction


### PostgreSQL in practice

- In the Stack Overflow Developer Survey 2024, PostgreSQL was the database that most of the responding developers used ([survey results, databases](https://survey.stackoverflow.co/2024/technology#most-popular-technologies-database)).
- The large cloud providers offer PostgreSQL as a managed service, for example Amazon RDS, Google Cloud SQL and Azure Database for PostgreSQL; a team then connects with a connection string and never installs a server.
- **Amazon Redshift**, a data warehouse, is based on PostgreSQL, so most of the SQL of this session runs there too.

> **Note.** A graphical client is useful for a first look at the tables and for trying a query. For work that must be repeated, write the query into a notebook or a script, as in this course.

## Creating tables and loading data: CREATE TABLE, INSERT, COPY

### `CREATE TABLE`, constraints and `INSERT`

`CREATE TABLE` defines a table: its name, its columns and its rules. Each column is written as `name type constraints`:

- the **name** of the column;
- the **type**, for example `INTEGER`, `NUMERIC(10, 2)`, `TEXT`, `DATE`, `BOOLEAN`;
- optional **constraints**, the rules every row must satisfy:
  - `PRIMARY KEY` declares the key (for a composite key, write `PRIMARY KEY (a, b)` after the columns);
  - `NOT NULL` forbids missing values;
  - `UNIQUE` forbids duplicates without making the column the key;
  - `CHECK (condition)` adds a rule for the values, for example `CHECK (price > 0)`;
  - `DEFAULT value` fills the column when an insert does not name it;
  - `REFERENCES other_table (column)` declares a foreign key.

`CREATE TABLE IF NOT EXISTS` does nothing when the table already exists, and `DROP TABLE name` deletes a table with all its rows.

`INSERT INTO table VALUES (...)` adds rows. A statement that violates a constraint fails with an error message, and the row is not stored.

*The column format `name type constraints` and the list of constraints follow [Data 100, SQL I](https://github.com/DS-100/fa23-course-notes/blob/main/sql_I/sql_I.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

### Why constraints matter

Constraints turn silent data problems into loud errors at the moment data are written. A rule written once in the schema protects every program that writes to the table, including the scripts of your teammates. PostgreSQL always checks every declared constraint, foreign keys included.

### Creating two small tables with constraints

The next cell creates the two small tables from the start of the notebook and fills them. They go into their own schema `demo`, so that they do not mix with the course tables. `DROP SCHEMA IF EXISTS demo CASCADE` removes an old copy with all its tables, so that the cell can run twice. One `execute` can run several statements separated by semicolons when it has no parameters. `--` starts a comment in SQL.

In [22]:
con.execute("""
DROP SCHEMA IF EXISTS demo CASCADE;
CREATE SCHEMA demo;
CREATE TABLE demo.listings (
    id        INTEGER PRIMARY KEY,                 -- one row per listing
    district  TEXT NOT NULL,
    room_type TEXT NOT NULL CHECK (room_type IN ('Entire home/apt', 'Private room', 'Hotel room', 'Shared room')),
    price     NUMERIC(10, 2) CHECK (price > 0)     -- NULL allowed: no price shown
);
CREATE TABLE demo.reviews (
    listing_id INTEGER NOT NULL REFERENCES demo.listings (id),   -- foreign key
    month      DATE NOT NULL,
    n_reviews  INTEGER NOT NULL DEFAULT 1 CHECK (n_reviews > 0),
    PRIMARY KEY (listing_id, month)                               -- composite key
);
INSERT INTO demo.listings VALUES (1, 'Mitte', 'Entire home/apt', 120), (2, 'Neukölln', 'Private room', 45),
                                 (3, 'Mitte', 'Entire home/apt', NULL);
INSERT INTO demo.reviews VALUES (1, '2026-04-01', 3), (1, '2026-05-01', 2), (2, '2026-05-01', 1);
""")
q("SELECT * FROM demo.listings")

,id,district,room_type,price
0,1,Mitte,Entire home/apt,120.0
1,2,Neukölln,Private room,45.0
2,3,Mitte,Entire home/apt,NaN


**Predict.** Which of these five inserts does the database accept, and which rule rejects each of the others?

In [23]:
invalid = {
    "unknown listing": "INSERT INTO demo.reviews VALUES (9, '2026-05-01', 1)",
    "same month twice": "INSERT INTO demo.reviews VALUES (1, '2026-05-01', 4)",
    "unknown room type": "INSERT INTO demo.listings VALUES (4, 'Mitte', 'Castle', 300)",
    "district missing": "INSERT INTO demo.listings (id, room_type) VALUES (5, 'Private room')",
    "month is not a date": "INSERT INTO demo.reviews VALUES (2, 'May 2026x', 1)",
}
for label, statement in invalid.items():
    try:
        con.execute(statement)
        print(f"{label:20} accepted")
    except psycopg.Error as error:
        print(f"{label:20} rejected ({type(error).__name__}): {error.diag.message_primary}")

unknown listing      rejected (ForeignKeyViolation): insert or update on table "reviews" violates foreign key constraint "reviews_listing_id_fkey"
same month twice     rejected (UniqueViolation): duplicate key value violates unique constraint "reviews_pkey"
unknown room type    rejected (CheckViolation): new row for relation "listings" violates check constraint "listings_room_type_check"
district missing     rejected (NotNullViolation): null value in column "district" of relation "listings" violates not-null constraint
month is not a date  rejected (InvalidDatetimeFormat): invalid input syntax for type date: "May 2026x"


The database rejects each invalid row. The message names the rule that failed, and the name of the error class says the kind of rule: `ForeignKeyViolation`, `UniqueViolation` (the composite primary key), `CheckViolation`, `NotNullViolation` and `InvalidDatetimeFormat`. PostgreSQL gives each constraint a name, such as `reviews_listing_id_fkey`; `CONSTRAINT name` in `CREATE TABLE` sets your own.

### More ways to change rows: `INSERT` variants, `UPDATE` and `DELETE`

- **With a column list**, `INSERT INTO reviews (listing_id, month) VALUES (2, '2026-06-01')`, the columns not named get their `DEFAULT` (here `n_reviews = 1`) or NULL. Naming the columns also keeps the statement correct if the table later gets a new column.
- **From a query**, `INSERT INTO table SELECT ...`, copies rows from another table.
- `UPDATE table SET column = value WHERE condition` changes existing rows, and `DELETE FROM table WHERE condition` removes rows. Without `WHERE`, both act on **every** row.
- `RETURNING columns` at the end of any of them returns the rows that the statement wrote, so that you see the result without a second query. `.fetchall()` returns these rows as a list of tuples.

In [24]:
print(con.execute("INSERT INTO demo.reviews (listing_id, month) VALUES (2, '2026-06-01') RETURNING *").fetchall())
print(con.execute("UPDATE demo.listings SET price = 50 WHERE id = 2 RETURNING id, price").fetchall())
print(con.execute("DELETE FROM demo.reviews WHERE month < '2026-05-01' RETURNING listing_id, month").fetchall())
q("SELECT * FROM demo.reviews ORDER BY listing_id, month")

[(2, datetime.date(2026, 6, 1), 1)]
[(2, 50.0)]
[(1, datetime.date(2026, 4, 1))]


,listing_id,month,n_reviews
0,1,2026-05-01,2
1,2,2026-05-01,1
2,2,2026-06-01,1


The new row took its `DEFAULT` of 1 review, listing 2 now costs 50, and the April row of listing 1 is gone. psycopg returns a date as a Python `datetime.date`.

### Loading many rows: executemany and commit

A loading script sends many rows at once. `cur.executemany(sql, rows)` runs one statement for every tuple in a list. The `%s` are **placeholders**: psycopg fills them with the values of each tuple (block 2 explains why values should always be passed this way). The next cell opens a second connection **without** `autocommit`: its statements form one **transaction**, and `commit()` ends it. Until then, the new rows are visible only to this connection and would be lost if the program stopped.

In [25]:
loader = psycopg.connect(server.get_uri("airbnb"))                       # no autocommit: one transaction
new_rows = [(4, "Pankow", "Private room", 60.0), (5, "Mitte", "Shared room", None)]
with loader.cursor() as cur:
    cur.executemany("INSERT INTO demo.listings VALUES (%s, %s, %s, %s)", new_rows)   # %s filled from each tuple
print("before commit, seen by con:", q("SELECT COUNT(*) FROM demo.listings").iloc[0, 0])
loader.commit()                                                                      # make the changes permanent
print("after commit, seen by con: ", q("SELECT COUNT(*) FROM demo.listings").iloc[0, 0])
loader.close()

before commit, seen by con: 3
after commit, seen by con:  5


Before the commit, the connection `con` still sees three listings; after it, five.

### Transactions: all or nothing

A transaction groups statements so that either all of them are stored or none. `with con.transaction():` opens one on any connection: at the end of the block PostgreSQL commits it, and if an error occurs inside the block, it rolls back every statement of the block. **Predict.** The block below inserts a valid listing 6 and then an invalid listing 7. Is listing 6 stored?

In [26]:
try:
    with con.transaction():
        con.execute("INSERT INTO demo.listings VALUES (6, 'Pankow', 'Private room', 60)")
        con.execute("INSERT INTO demo.listings VALUES (7, 'Pankow', 'Castle', 60)")      # breaks the CHECK
except psycopg.errors.CheckViolation as error:
    print("rolled back:", error.diag.message_primary)
q("SELECT id FROM demo.listings ORDER BY id")

rolled back: new row for relation "listings" violates check constraint "listings_room_type_check"


,id
0,1
1,2
2,3
3,4
4,5


Listing 6 is not stored: the error in the second statement undid the first one. A loading script uses one transaction for the whole load, so that a failed load leaves no half-loaded table.

### `COPY`: loading a file in one statement

`COPY table (columns) FROM ... (FORMAT csv, HEADER)` is PostgreSQL's command for loading many rows at once; it is much faster than one `INSERT` per row. It comes in two forms:

- `COPY ... FROM '/path/file.csv'` reads a file **on the server's computer** and needs special rights;
- `COPY ... FROM STDIN` reads data that the **client sends**. In Python, `cur.copy(...)` opens such a copy and `copy.write(data)` sends the data. In `psql`, the command `\copy` does the same.

The idea is the same as for any load: create the table with its rules first, load all rows in one statement or transaction, then compare the row count with the source. The build script loads the course data this way. Here 200 listings are written to a CSV file in a temporary folder (the module `tempfile` keeps it out of the repository), as if it came from a colleague, and loaded with `COPY ... FROM STDIN`. An empty field in a CSV file becomes NULL.

In [27]:
import tempfile

con.execute("""CREATE TABLE demo.listings_loaded (id BIGINT PRIMARY KEY, district TEXT NOT NULL,
                                                 room_type TEXT NOT NULL, price NUMERIC(10, 2) CHECK (price > 0))""")
sample = q("SELECT id, district, room_type, price FROM listings ORDER BY id LIMIT 200")
csv_path = Path(tempfile.mkdtemp()) / "listings_sample.csv"
sample.to_csv(csv_path, index=False)

with con.cursor() as cur:
    with cur.copy("COPY demo.listings_loaded FROM STDIN (FORMAT csv, HEADER)") as copy:
        copy.write(csv_path.read_text(encoding="utf-8"))
print("rows in file:", len(sample), "| rows in table:", q("SELECT COUNT(*) FROM demo.listings_loaded").iloc[0, 0])

rows in file: 200 | rows in table: 200


**Predict.** What happens when the same file is loaded a second time?

In [28]:
try:
    with con.cursor() as cur:
        with cur.copy("COPY demo.listings_loaded FROM STDIN (FORMAT csv, HEADER)") as copy:
            copy.write(csv_path.read_text(encoding="utf-8"))
except psycopg.errors.UniqueViolation as error:
    print("second load refused:", error.diag.message_primary)
    print(error.diag.message_detail)
print("rows in table:", q("SELECT COUNT(*) FROM demo.listings_loaded").iloc[0, 0])

second load refused: duplicate key value violates unique constraint "listings_loaded_pkey"
Key (id)=(3176) already exists.
rows in table: 200


A `COPY` is one statement, so it is stored completely or not at all: the first duplicate id stops it, and the table keeps its 200 rows. The `DETAIL` line names the duplicate key.

#### Loading a DataFrame with `to_sql`

pandas can write a DataFrame into a table with `DataFrame.to_sql(name, engine, schema=..., if_exists="append", index=False)`. For PostgreSQL, pandas needs a SQLAlchemy **engine**, an object that opens connections from a connection string; `sqlalchemy.create_engine("postgresql+psycopg://...")` creates one that uses psycopg. Loading the 200 rows again breaks the primary key, and the whole batch is refused. pandas wraps the error in its own `pd.errors.DatabaseError`; `error.__cause__` is the error of SQLAlchemy, and its `.orig` the error of PostgreSQL:

In [29]:
import sqlalchemy

engine = sqlalchemy.create_engine(server.get_uri("airbnb").replace("postgresql://", "postgresql+psycopg://"))
try:
    sample.to_sql("listings_loaded", engine, schema="demo", if_exists="append", index=False)
except pd.errors.DatabaseError as error:                  # pandas wraps the database error
    print("second load refused:", error.__cause__.orig.diag.message_primary)   # the original PostgreSQL message
print("rows in table:", q("SELECT COUNT(*) FROM demo.listings_loaded").iloc[0, 0])

error ignored terminating <psycopg.Pipeline [INERROR, pipeline=ON] (user=postgres database=airbnb) at 0x11f3ef890>: pipeline aborted


second load refused: duplicate key value violates unique constraint "listings_loaded_pkey"
rows in table: 200


> **Note.** `DataFrame.to_sql(..., if_exists="replace")` drops the table and recreates it **without** primary keys or constraints. To keep them, create the table with `CREATE TABLE` first and load with `if_exists="append"` or `COPY`, as [`make_airbnb_postgres.py`](make_airbnb_postgres.py) does.

### Schema first: what strict rules find in real data

The course tables have types and primary keys, but few other rules. `INSERT INTO table SELECT ...` copies the result of a query into a table, so the real listings can be tested against stricter rules: a known room type, a minimum stay of at least one night, a maximum stay below 2,147,483,647 (the largest 32-bit integer), and a minimum stay that is not longer than the maximum stay. A `CHECK` that names two columns is written after the columns.

In [30]:
con.execute("""
CREATE TABLE demo.listings_strict (
    id             BIGINT PRIMARY KEY,
    room_type      TEXT NOT NULL CHECK (room_type IN ('Entire home/apt', 'Private room', 'Hotel room', 'Shared room')),
    minimum_nights INTEGER CHECK (minimum_nights >= 1),
    maximum_nights INTEGER CHECK (maximum_nights < 2147483647),
    CHECK (minimum_nights <= maximum_nights)
)""")
try:
    con.execute("INSERT INTO demo.listings_strict SELECT id, room_type, minimum_nights, maximum_nights FROM listings")
except psycopg.errors.CheckViolation as error:
    print("rejected:", error.diag.message_primary)
    print(error.diag.message_detail)
print("listings stored:", q("SELECT COUNT(*) FROM demo.listings_strict").iloc[0, 0])

rejected: new row for relation "listings_strict" violates check constraint "listings_strict_maximum_nights_check"
Failing row contains (28004334, Hotel room, 1, 2147483647).
listings stored: 0


The database refuses the whole statement, and because it was one statement, not a single listing was stored. The message names the constraint that failed, and the `DETAIL` line shows the first row that broke it. Which rows break the rule?

In [31]:
q("SELECT id, minimum_nights, maximum_nights FROM listings WHERE maximum_nights >= 2147483647")

,id,minimum_nights,maximum_nights
0,28004334,1,2147483647
1,50161791,1,2147483647


Two listings carry the maximum stay 2,147,483,647: a software default, not a stay. The loader must now **decide** (set it to NULL, or drop the listings), and write the decision down. Extra 02 makes these decisions for all three tables and checks the counts against the source files; it is the model for the team project's loader.

*This subsection is a short version of [extras/02-loading-with-constraints.ipynb](extras/02-loading-with-constraints.ipynb) (course team, CC BY 4.0).*

### Loading data in practice

- **Data engineering teams** keep the `CREATE TABLE` statements of a project in version-controlled `.sql` files and review changes in pull requests, like any other code.
- **Public Health England** lost about 16,000 positive COVID-19 test results in October 2020 because a manual step used an old Excel file format whose row limit was exceeded (BBC News, 5 October 2020); a loading script with row-count checks would have caught it.

> 📖 **Read more:** [extras/02-loading-with-constraints.ipynb](extras/02-loading-with-constraints.ipynb) (course team): a strict course database with keys and checks for all three tables, the rows the rules reject, documented loading decisions and counts against the source files.

The demonstration tables are no longer needed:

In [32]:
engine.dispose()                              # close the connections of the SQLAlchemy engine
con.execute("DROP SCHEMA demo CASCADE")      # removes the schema demo with all its tables
print("schema demo removed")

schema demo removed


## Queries: SELECT, WHERE, ORDER BY, LIMIT

### The basic clauses

A query describes the result you want, not the steps to compute it; the database chooses how. SQL is therefore called a **declarative** language, unlike Python, where you write the steps. The basic clauses are:

- `SELECT` names the columns (or expressions) of the result; `*` means all columns; `AS` gives a result column a new name (an **alias**);
- `FROM` names the table;
- `WHERE` gives a condition that each row must satisfy;
- `ORDER BY` sorts the result (`ASC` ascending is the default, `DESC` descending);
- `LIMIT n` keeps only the first *n* rows; `OFFSET k` skips the first *k* rows before that.

Every query needs `SELECT` and (to read a table) `FROM`, and the clauses must be written in this order:

```
SELECT <column list>
FROM <table>
[WHERE <condition>]
[ORDER BY <column list>]
[LIMIT <number of rows>]
[OFFSET <number of rows>];
```

Square brackets mark optional clauses. By convention SQL keywords are written in capitals and each clause on its own line; SQL ignores upper and lower case in keywords and ignores line breaks, so both are only for the reader. A semicolon ends a statement; psycopg accepts a query with or without it. PostgreSQL also accepts the standard form `FETCH FIRST 5 ROWS ONLY` for `LIMIT 5`.

*The clause template and conventions adapted from [Data 100, SQL I](https://github.com/DS-100/fa23-course-notes/blob/main/sql_I/sql_I.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

### The logical order of the clauses

Although we write `SELECT` first, the database evaluates the clauses in a different order:

```mermaid
flowchart LR
    F["1 FROM"] --> W["2 WHERE"] --> G["3 GROUP BY"] --> H["4 HAVING"] --> S["5 SELECT"] --> O["6 ORDER BY"] --> L["7 LIMIT / OFFSET"]
```

`GROUP BY` and `HAVING` follow in block 2. The next subsection shows what this order means for an alias defined with `AS` in `SELECT`.

By hand: in the small tables from the start of the notebook, `SELECT id FROM listings WHERE district = 'Mitte' ORDER BY id` keeps listings 1 and 3 and returns them in this order.

#### A `SELECT` alias in `WHERE` and in `ORDER BY`

The logical order explains a common error. An alias is created in step 5, `SELECT`. When `WHERE` runs in step 2, the alias does not exist yet; when `ORDER BY` runs in step 6, it does. In general form:

```sql
SELECT name, price * quantity AS total
FROM orders
WHERE total > 100;        -- ERROR: column "total" does not exist
```

```sql
SELECT name, price * quantity AS total
FROM orders
ORDER BY total DESC;      -- fine: ORDER BY runs after SELECT
```

**Predict.** On the listings, which line does the error message point to?

In [33]:
try:
    q("""
    SELECT id, price, ROUND(price / accommodates, 1) AS price_per_guest
    FROM listings
    WHERE price_per_guest > 50
    """)
except psycopg.errors.UndefinedColumn as error:
    print(error)

column "price_per_guest" does not exist
LINE 4:     WHERE price_per_guest > 50
                  ^


`column "price_per_guest" does not exist` names the alias, and `LINE 4:` with the `^` points to the `WHERE` line (line 1 of the query text is the empty line after the opening quotes). To filter on a computed value, repeat the expression in `WHERE`, or compute the column in a subquery first (block 2). In `ORDER BY`, step 6, the alias exists:

In [34]:
q("""
SELECT id, price, accommodates, ROUND(price / accommodates, 1) AS price_per_guest
FROM listings
WHERE price / accommodates > 50                -- repeat the expression
ORDER BY price_per_guest DESC                  -- the alias exists here: ORDER BY runs after SELECT
LIMIT 3
""")

,id,price,accommodates,price_per_guest
0,1704420758533033977,7999.20,5,1599.8
1,1043552887218608966,10025.00,7,1432.1
2,1320202792638588083,759.57,1,759.6


The highest prices per guest are far above a normal holiday rental; Session 4 checks such values.

> **Note.** `ROUND(x, 1)` works here because `price` is `NUMERIC`. For a `DOUBLE PRECISION` column such as `latitude`, PostgreSQL has only `ROUND(x)` without digits; write `ROUND(latitude::numeric, 2)`.

### Why filtering in the database matters

Filtering in the database moves only the rows you need into Python. With 3.8 million calendar rows, `SELECT *` followed by filtering in pandas wastes memory and time. Writing the condition out also makes the question precise: "affordable short stays for a family" becomes `room_type = 'Entire home/apt' AND minimum_nights < 28 AND accommodates >= 4 AND price <= 150`.

### SELECT: columns, aliases, expressions and DISTINCT

Columns appear in the order you name them. An expression such as `price / accommodates` computes a new column for every row; `ROUND(x, 1)` rounds to one decimal, and `AS` names the result.

In [35]:
q("""
SELECT district, room_type, accommodates AS guests, price,
       ROUND(price / accommodates, 1) AS price_per_guest
FROM listings
ORDER BY id
LIMIT 5
""")

,district,room_type,guests,price,price_per_guest
0,Pankow,Entire home/apt,2,160.71,80.4
1,Pankow,Entire home/apt,7,193.33,27.6
2,Pankow,Entire home/apt,1,NaN,NaN
3,Friedrichshain-Kreuzberg,Entire home/apt,2,10.88,5.4
4,Pankow,Entire home/apt,4,372.67,93.2


`SELECT DISTINCT` removes duplicate rows from the result. With two columns, it keeps each different *combination* once. A subquery after `FROM` needs no name in PostgreSQL 16 and newer; older versions require one, as in `FROM (...) AS t`.

In [36]:
print(q("SELECT DISTINCT room_type FROM listings ORDER BY room_type"))
q("SELECT COUNT(*) AS n_combinations FROM (SELECT DISTINCT district, room_type FROM listings)")

         room_type
0  Entire home/apt
1       Hotel room
2     Private room
3      Shared room


,n_combinations
0,38


### WHERE: conditions

`WHERE` keeps the rows for which the condition is true. Conditions use the comparison operators `=`, `<>` (or `!=`), `<`, `<=`, `>`, `>=` and combine with `AND`, `OR` and `NOT`. Text values are written in single quotes: `'Mitte'`. Double quotes are for names of columns and tables only: PostgreSQL reads `"Mitte"` as a column called Mitte.

The best-rated entire flats in Neukölln under €100 a night that can be booked for less than 28 nights and have at least 50 reviews:

In [37]:
q("""
SELECT id, neighbourhood, price, number_of_reviews, review_scores_rating
FROM listings
WHERE district = 'Neukölln' AND room_type = 'Entire home/apt'
  AND minimum_nights < 28 AND price < 100 AND number_of_reviews >= 50    -- all conditions must hold
ORDER BY review_scores_rating DESC, number_of_reviews DESC             -- best rated first; reviews break ties
LIMIT 3
""")

,id,neighbourhood,price,number_of_reviews,review_scores_rating
0,31382644,Schillerpromenade,92.5,374,4.91
1,237670,Buckow Nord,68.5,91,4.84
2,190448,Buckow Nord,85.0,87,4.77


#### `AND`, `OR` and parentheses

**Predict.** `AND` binds more strongly than `OR`, as multiplication binds more strongly than addition. The two counts below differ only in the parentheses. Which is larger, and which one answers "private rooms in Mitte or Pankow"?

In [38]:
q("""
SELECT
  (SELECT COUNT(*) FROM listings
   WHERE room_type = 'Private room' AND district = 'Mitte' OR district = 'Pankow')   AS without_parentheses,
  (SELECT COUNT(*) FROM listings
   WHERE room_type = 'Private room' AND (district = 'Mitte' OR district = 'Pankow')) AS with_parentheses
""")

,without_parentheses,with_parentheses
0,2144,911


Without parentheses, the condition reads "(private room in Mitte) or (anything in Pankow)". Write the parentheses whenever `AND` and `OR` meet.

#### Shorter conditions: `IN`, `BETWEEN` and `NOT`

`IN (...)` tests membership in a list (like Python's `in`), `BETWEEN a AND b` tests a range with both ends included, and `NOT` negates either.

In [39]:
q("""
SELECT district, room_type, price
FROM listings
WHERE district IN ('Mitte', 'Pankow') AND room_type NOT IN ('Entire home/apt')
  AND price BETWEEN 50 AND 60              -- same as price >= 50 AND price <= 60
ORDER BY price, id
LIMIT 5
""")

,district,room_type,price
0,Pankow,Private room,50.00
1,Mitte,Private room,50.00
2,Pankow,Private room,50.84
3,Pankow,Hotel room,51.00
4,Pankow,Private room,51.30


#### Text patterns with `LIKE` and `ILIKE`

In a `LIKE` pattern, `%` stands for any number of characters (also none) and `_` for exactly one character. In PostgreSQL, `LIKE` is **case-sensitive**; `ILIKE` ignores upper and lower case. **Predict.** How many listing titles contain "balcony" with `LIKE`, and how many with `ILIKE`?

A condition is a `boolean`, so `COUNT(*) FILTER (WHERE condition)` counts the rows where it is true. `FILTER` is part of the SQL standard and lets one query compute several counts.

In [40]:
q("""
SELECT COUNT(*) FILTER (WHERE name LIKE '%balcony%')                          AS like_lower_case,
       COUNT(*) FILTER (WHERE name ILIKE '%balcony%')                         AS ilike_any_case,
       COUNT(*) FILTER (WHERE name ILIKE '%balcony%' OR name ILIKE '%balkon%') AS balcony_or_balkon
FROM listings
""")

,like_lower_case,ilike_any_case,balcony_or_balkon
0,229,412,583


`LIKE` finds only the titles with a lower-case "balcony". `ILIKE` also finds "Balcony" and "BALCONY", and with the German "balkon" 583 titles mention a balcony. `ILIKE` also ignores the case of letters such as `ö`:

In [41]:
q("""
SELECT COUNT(*) FILTER (WHERE district ILIKE 'NEUKÖLLN') AS ilike_capital_umlaut,
       COUNT(*) FILTER (WHERE district LIKE 'NEUKÖLLN')  AS like_capital_umlaut,
       upper('neukölln')                                AS upper_case
FROM listings
""")

,ilike_capital_umlaut,like_capital_umlaut,upper_case
0,1046,0,NEUKÖLLN


`_` matches exactly one character. Some neighbourhoods in Reinickendorf are called "West 1" to "West 5", and Charlottenburg has a "Westend". **Predict.** Which of them does `'West _'` match?

In [42]:
q("SELECT DISTINCT neighbourhood FROM listings WHERE neighbourhood LIKE 'West _' ORDER BY neighbourhood")

,neighbourhood
0,West 1
1,West 2
2,West 3
3,West 4
4,West 5


`'West _'` needs exactly one character after the space, so it keeps "West 1" to "West 5" and leaves out "Westend". `'West%'` would match all six.

*`LIKE` with `%` and `_` adapted from [Data 100, SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (UC Berkeley Data 100, BSD 3-Clause), here on the Airbnb data.*

#### Reading `LIKE` patterns

A pattern describes the whole text, from the first to the last character. Without `%` at the start or end, the text must start or end exactly as written.

| Pattern | Means | Matches | Does not match |
|---|---|---|---|
| `LIKE 'West%'` | starts with "West" | West 1, Westend | Nord 1 |
| `LIKE '%end'` | ends with "end" | Westend | West 1 |
| `ILIKE '%balkon%'` | contains "balkon" anywhere, in any case | Balkon, BALKON, Südbalkon | Balcony |
| `LIKE 'West _'` | "West", a space, then exactly one character | West 1 | Westend, West 10 |
| `LIKE '_____'` | exactly five characters | Mitte | Pankow |
| `LIKE 'Mitte'` | exactly "Mitte" (no wildcard: like `=`) | Mitte | MITTE, Mitte-Nord |

#### `NOT LIKE`

`NOT LIKE` and `NOT ILIKE` keep the rows that do not match. The titles with a balcony and those without should add up to all listings:

In [43]:
q("""
SELECT COUNT(*) AS all_listings,
       COUNT(*) FILTER (WHERE name ILIKE '%balcony%' OR name ILIKE '%balkon%')          AS with_balcony,
       COUNT(*) FILTER (WHERE name NOT ILIKE '%balcony%' AND name NOT ILIKE '%balkon%') AS without_balcony
FROM listings
""")

,all_listings,with_balcony,without_balcony
0,10221,583,9638


583 + 9,638 = 10,221: here they add up, because every listing has a title. If `name` were NULL, both `ILIKE` and `NOT ILIKE` would be unknown and the row would fall out of both counts (the section on NULL values explains why). Note the `AND` in the second condition: "neither balcony nor balkon".

#### Searching for `%` or `_` themselves: `ESCAPE`

To search for `%` or `_` themselves, put the **escape character** in front of them. In PostgreSQL the escape character is the backslash `\` by default, and `ESCAPE '!'` sets another one. In Python, write a query with a backslash as a raw string `r"""..."""`, so that Python keeps the backslash as it is.

**Predict.** `name LIKE '%50%%'` is meant to find titles that say "50%". How many does it find, and what do they contain?

In [44]:
q(r"""
SELECT COUNT(*) FILTER (WHERE name LIKE '%50%%')            AS percent_as_wildcard,
       COUNT(*) FILTER (WHERE name LIKE '%50\%%')           AS literal_50_percent,
       COUNT(*) FILTER (WHERE name LIKE '%!%%' ESCAPE '!')  AS any_percent_sign
FROM listings
""")

,percent_as_wildcard,literal_50_percent,any_percent_sign
0,46,0,2


The first pattern finds 46 titles, but they contain things like "50 m²" or "150 qm": the middle `%` is a wildcard, so the pattern only asks for "50" somewhere. With the escape character, `\%` means a real percent sign: no title says "50%", and two titles contain a percent sign at all.

#### Regular expressions with `~`

A **regular expression** is a pattern language that is much richer than `LIKE`: it can say "one or more digits", "this word or that word" or "a word boundary". Python's module `re` uses almost the same language, and PostgreSQL has it built in:

| Operator | Means |
|---|---|
| `text ~ 'pattern'` | the pattern occurs, case-sensitive |
| `text ~* 'pattern'` | the pattern occurs, ignoring case |
| `text !~ 'pattern'`, `text !~* 'pattern'` | the pattern does not occur |
| `regexp_replace(text, 'pattern', 'new')` | replaces the matches |

Unlike `LIKE`, a regular expression matches anywhere in the text unless it says otherwise. How many titles give the size of the flat, such as "56 m²", "130 sqm" or "80qm"? The pattern `\y[0-9]+ ?(sqm|qm|m²)` reads: a word boundary `\y`, one or more digits `[0-9]+`, an optional space ` ?`, and one of three units `(sqm|qm|m²)`. There is one trap: a word boundary is `\y` in PostgreSQL, while `\b`, the word boundary in Python, means the backspace character.

In [45]:
q(r"""
SELECT COUNT(*) FILTER (WHERE name ~ '\y[0-9]+ ?(sqm|qm|m²)')  AS titles_with_size,
       COUNT(*) FILTER (WHERE name ~ '\b[0-9]+ ?(sqm|qm|m²)')  AS with_python_boundary,
       COUNT(*) FILTER (WHERE name ~* 'balcony|balkon')         AS with_balcony
FROM listings
""")

,titles_with_size,with_python_boundary,with_balcony
0,459,0,583


459 titles state a size. With `\b` the pattern finds nothing, and PostgreSQL gives no warning: test a pattern copied from Python. `~*` finds the same 583 balcony titles as the two `ILIKE` conditions, with one condition: `|` means "or". Use `LIKE` or `ILIKE` when they can express the pattern, because they are easier to read, and a regular expression when they cannot. (The standard operator `SIMILAR TO` also exists, but it mixes both pattern languages and is rarely used.)

#### Conditions on dates

`calendar.date` is a `date`, so a range is written with date literals: `date BETWEEN DATE '2026-08-01' AND DATE '2026-08-31'`. `EXTRACT(DOW FROM date)` returns the weekday as a number (0 is Sunday, 6 Saturday). **Predict.** `available` is a `boolean`. Does `AVG(available)` give the share of available nights?

In [46]:
try:
    q("SELECT AVG(available) FROM calendar")
except psycopg.errors.UndefinedFunction as error:
    print(error.diag.message_primary, "|", error.diag.message_hint)

function avg(boolean) does not exist | No function matches the given name and argument types. You might need to add explicit type casts.


PostgreSQL has no average of true/false values, and the hint suggests a cast: `available::int` turns `true` into 1 and `false` into 0, and the average of the 1s and 0s is the share of available nights. The share of Saturday nights in August that are still available in the calendar:

In [47]:
q("""
SELECT COUNT(*) AS saturday_nights, ROUND(AVG(available::int), 3) AS share_available
FROM calendar
WHERE date BETWEEN DATE '2026-08-01' AND DATE '2026-08-31' AND EXTRACT(DOW FROM date) = 6
""")

,saturday_nights,share_available
0,51500,0.389


A night that is not available is booked or blocked by the host; the calendar does not say which.

### ORDER BY, LIMIT and OFFSET

`ORDER BY a, b` sorts by `a` and uses `b` to break ties; each column can have its own `DESC`. `LIMIT 3 OFFSET 3` returns rows 4 to 6 of the sorted result, which is how a website shows "page 2". The longest minimum stays:

In [48]:
q("""
SELECT id, district, room_type, minimum_nights
FROM listings
ORDER BY minimum_nights DESC NULLS LAST, id
LIMIT 3 OFFSET 0       -- try OFFSET 3 for the next page
""")

,id,district,room_type,minimum_nights
0,6670861,Neukölln,Entire home/apt,1125
1,584757,Pankow,Private room,1000
2,6704144,Pankow,Private room,1000


A minimum stay of 1,125 nights, three years, is not a holiday rental; Session 4 treats such values. `NULLS LAST` is needed here: in descending order, PostgreSQL puts NULL **first**, before every number, and two listings have no minimum stay. The section on NULL values explains the sort order.

> **Note.** Without `ORDER BY`, the order of rows is not defined. `LIMIT 10` without `ORDER BY` returns *some* ten rows, which may change between runs or database versions.

### CASE: a value that depends on a condition

`CASE WHEN condition THEN value ... ELSE value END` computes a value row by row, like `if`/`elif`/`else` in Python. The first true condition wins. Here each listing is put into a stay category; block 2 counts listings per category with `GROUP BY`.

In [49]:
q("""
SELECT id, minimum_nights,
       CASE WHEN minimum_nights < 7  THEN 'short stay'
            WHEN minimum_nights < 28 THEN 'week or more'
            ELSE 'month or more'
       END AS stay_type
FROM listings
ORDER BY id
LIMIT 5
""")

,id,minimum_nights,stay_type
0,3176,2,short stay
1,9991,6,short stay
2,14325,180,month or more
3,17904,93,month or more
4,20858,3,short stay


*`CASE` adapted from [Data 100, SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

### Queries in practice

- **Property portals** turn the search form (district, number of rooms, maximum rent) into exactly such filter conditions before they sort the result by date or price, and use `LIMIT`/`OFFSET` for the result pages.
- **Data engineers** check every nightly load with small queries such as "how many orders arrived yesterday?".

> 📖 **Read more:** [PostgreSQL tutorial, chapter 2: The SQL language](https://www.postgresql.org/docs/current/tutorial-sql.html) (PostgreSQL Global Development Group): creating tables, filling them and querying them, step by step on a small weather database.

## NULL values

### NULL and three-valued logic

**NULL** means "missing" or "unknown". It is not zero and not the empty text `''`. SQL treats it with **three-valued logic**: a condition is true, false or unknown.

- Any comparison with NULL is unknown: `price > 100` is unknown for a listing without a price, and so is `price = NULL`, and even `NULL = NULL`. `WHERE` keeps only rows whose condition is *true*, so rows with unknown conditions disappear from both `WHERE price > 100` and `WHERE NOT (price > 100)`.
- To test for NULL, write `price IS NULL` or `price IS NOT NULL`. To compare two values and treat two NULLs as equal, write `a IS NOT DISTINCT FROM b`; `a IS DISTINCT FROM b` is its opposite.
- Arithmetic and text operations with NULL give NULL: `price * 2` is NULL when the price is, and so is `'Berlin' || NULL`.
- `COALESCE(price, 0)` returns the first value that is not NULL, here 0 for missing prices. Use it only when "missing" really means zero.
- Counting: `COUNT(*)` counts rows; `COUNT(price)` counts only the rows where `price` is not NULL. Functions such as `AVG(price)` and `SUM(price)` skip NULLs.
- Sorting: PostgreSQL sorts NULL after every other value: last in ascending order, first in descending order. `ORDER BY price NULLS LAST` or `NULLS FIRST` makes the order explicit. Other databases differ (SQLite and MySQL put NULL first in ascending order), so write it out when the order matters.

| `a` | `b` | `a AND b` | `a OR b` | `NOT a` |
|---|---|---|---|---|
| true | unknown | unknown | true | false |
| false | unknown | false | unknown | true |
| unknown | unknown | unknown | unknown | unknown |

By hand, with the three small listings from the start of the notebook: `WHERE price < 100` keeps only listing 2; `WHERE price >= 100` keeps only listing 1; listing 3 is in neither result. `COUNT(*)` is 3, `COUNT(price)` is 2, and the average price is (120 + 45) / 2 = 82.5, not (120 + 45 + 0) / 3 = 55.

### NULL in conditions and counts

Every listing is above €150, not above €150, or without a price. **Predict.** Do the first two counts add up to all listings?

In [50]:
q("""
SELECT (SELECT COUNT(*) FROM listings WHERE price > 150)          AS above_150,
       (SELECT COUNT(*) FROM listings WHERE NOT (price > 150))    AS not_above_150,
       (SELECT COUNT(*) FROM listings WHERE price IS NULL)        AS no_price,
       (SELECT COUNT(*) FROM listings WHERE price = NULL)         AS equals_null,
       (SELECT COUNT(*) FROM listings)                            AS all_listings
""")

,above_150,not_above_150,no_price,equals_null,all_listings
0,2846,3908,3467,0,10221


Only the first three numbers together give all listings: the listings without a price fall out of both the condition and its negation, and only `IS NULL` finds them. `price = NULL` finds nothing, because the comparison is never true.

### NULL in expressions, `COALESCE` and the sort order

**Predict.** In descending order of price, do the listings without a price come first or last?

In [51]:
q("""
SELECT id, room_type, price, price * 2 AS double_price, COALESCE(price, -1) AS price_or_minus_one,
       price IS DISTINCT FROM 120 AS differs_from_120
FROM listings
WHERE district = 'Mitte'
ORDER BY price DESC          -- descending: PostgreSQL puts the NULLs first; try ORDER BY price DESC NULLS LAST
LIMIT 3
""")

,id,room_type,price,double_price,price_or_minus_one,differs_from_120
0,191023,Private room,None,None,-1.0,True
1,251406,Entire home/apt,None,None,-1.0,True
2,499342,Entire home/apt,None,None,-1.0,True


In descending order the listings without a price come first, and `price * 2` stays NULL. `price IS DISTINCT FROM 120` is `True` for a missing price, while `price <> 120` would be NULL.

### Is a missing price random?

A listing whose calendar has no available night in the coming year (`availability_365 = 0`) cannot show the price of a next free night. `COUNT(*) FILTER (WHERE ...)` gives the three counts in one pass over the table.

In [52]:
q("""
SELECT COUNT(*) FILTER (WHERE price IS NULL)                              AS no_price,
       COUNT(*) FILTER (WHERE price IS NULL AND availability_365 = 0)     AS no_price_no_free_night,
       COUNT(*) FILTER (WHERE price IS NOT NULL AND availability_365 = 0) AS price_no_free_night
FROM listings
""")

,no_price,no_price_no_free_night,price_no_free_night
0,3467,2744,1


Four in five listings without a price have no free night at all: they are probably not offered at the moment. Any statement about "the price of a listing in Berlin" therefore describes the listings that can be booked, not all listings. Session 4 returns to such missing values.

### NULL in practice

- **Surveys** distinguish "answered 0" from "did not answer"; storing both as 0 makes the average wrong.
- **Health records** leave a laboratory value empty when the test was not done; treating empty as "normal" or as zero leads to wrong conclusions.

> **Note.** `WHERE x NOT IN (...)` behaves unexpectedly when the list contains a NULL. Block 2 shows safer ways to find rows without a partner.

## Worked example · Which flat can a family book for a weekend?

A family of four looks for an entire flat in Friedrichshain-Kreuzberg for a weekend: a minimum stay of at most two nights, at least four guests, at most €150 a night, and at least 20 reviews.

### Step 1 · Turn the family's wishes into conditions and a sort order

Each part of the question becomes one condition; the best-rated flats come first, and the number of reviews breaks ties. `NULLS LAST` keeps flats without a rating at the end.

**Why this step?** Why sort by rating and then by number of reviews, and not by rating alone?

In [53]:
family = q("""
SELECT id, neighbourhood, accommodates, price, number_of_reviews, review_scores_rating
FROM listings
WHERE district = 'Friedrichshain-Kreuzberg'
  AND room_type = 'Entire home/apt'
  AND minimum_nights <= 2 AND accommodates >= 4 AND price <= 150 AND number_of_reviews >= 20
ORDER BY review_scores_rating DESC NULLS LAST, number_of_reviews DESC
LIMIT 5
""")
family

,id,neighbourhood,accommodates,price,number_of_reviews,review_scores_rating
0,20660494,Tempelhofer Vorstadt,4,128.67,361,4.88
1,51830435,Frankfurter Allee Süd FK,6,128.00,271,4.86
2,48732044,Karl-Marx-Allee-Süd,6,129.00,298,4.85
3,15887444,Südliche Friedrichstadt,4,125.00,23,4.83
4,26633041,Tempelhofer Vorstadt,4,92.33,372,4.80


### Step 2 · Check the calendar of the best flat

Is the best-rated flat still free for the first weekend of August 2026? We take its id from the result and pass it to the calendar query with a `%s` placeholder (block 2 explains why this is safer than pasting the id into the text). `to_char(date, 'Dy')` writes the weekday as a short name.

> **Note.** In a query with parameters, psycopg reads every `%` as the start of a placeholder; write `%%` for a `LIKE` wildcard.

**Why this step?** The flat passed every condition of step 1. Why look at a second table at all?

In [54]:
best = int(family.loc[0, "id"])
q("""
SELECT date, to_char(date, 'Dy') AS weekday, available
FROM calendar
WHERE listing_id = %s AND date BETWEEN DATE '2026-07-31' AND DATE '2026-08-02'
ORDER BY date
""", params=(best,))

,date,weekday,available
0,2026-07-31,Fri,False
1,2026-08-01,Sat,False
2,2026-08-02,Sun,False


### Step 3 · Read the answer and decide

In this snapshot all three nights are unavailable, so the family moves on to the second flat of the list. The answer comes from two small queries on two tables, linked by the key `id` = `listing_id`. Block 2 does such links in one query with a join.

## Check your understanding

Vote alone first, discuss with your neighbour, then vote again. Each question has one best answer.

**1.** In PostgreSQL, `reviews.listing_id` is declared with `REFERENCES listings (id)`. What happens to `INSERT INTO reviews VALUES (9, '2026-05-01', 1)` when there is no listing 9?

- **A.** The row is stored, and a later join drops it.
- **B.** The row is stored with `listing_id` set to NULL.
- **C.** The insert fails with a foreign key violation, and nothing is stored.
- **D.** The insert fails only if foreign key checks were switched on for the connection first.

**2.** `first_review` is a `DATE`, and one listing has `first_review = DATE '2026-06-20'`. What does `DATE '2026-06-26' - first_review` return for it?

- **A.** The integer 6, the number of days.
- **B.** The text `'0000-00-06'`.
- **C.** An error: dates cannot be subtracted; they must first be converted to numbers.
- **D.** The timestamp `2026-06-06 00:00`.

**3.** In which order does the database evaluate the clauses of `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT`?

- **A.** `FROM`, `WHERE`, `SELECT`, `ORDER BY`, `LIMIT`
- **B.** `SELECT`, `FROM`, `WHERE`, `ORDER BY`, `LIMIT`
- **C.** `FROM`, `SELECT`, `WHERE`, `LIMIT`, `ORDER BY`
- **D.** In the order they are written; the optimiser only makes it faster.

**4.** 3,467 listings have no price, yet `SELECT COUNT(*) FROM listings WHERE price = NULL` returns 0. Why?

- **A.** The missing prices are stored as 0, so they never equal NULL.
- **B.** `COUNT(*)` skips the rows in which a column is NULL.
- **C.** A comparison with NULL is unknown for every row, and `WHERE` keeps only the rows whose condition is true.
- **D.** NULL is stored as the text `'NULL'`, so the comparison needs quotes.

**5.** A CSV file of 200 listings is loaded with `COPY demo.listings_loaded FROM STDIN (FORMAT csv, HEADER)`. Row 150 has an `id` that is already in the table. What is stored?

- **A.** 199 rows: `COPY` skips the row that breaks the key and loads the rest.
- **B.** 149 rows: `COPY` stops at row 150 and keeps the rows before it.
- **C.** No new row: the whole `COPY` fails, because it is one statement.
- **D.** 200 rows: `COPY` does not check constraints, so that it can load fast.

## Exercises

Work in pairs; the lecturer picks which exercises to set. The cells above have opened the course database as `con` and defined `q(sql, params)`, which returns a query result as a DataFrame. For **Paper** and **Predict** tasks, write your answer in the markdown cell before you run anything. *Stretch* tasks are for pairs that finish early.

### 1.1 · Warm-ups: predict the output

*Predict*

> What does the database return?
> 1. `SELECT 1/2;`
> 2. `SELECT NULL = NULL;`
> 3. `SELECT 'Berlin' || NULL;`
> 4. `SELECT COUNT(*), COUNT(price), AVG(price) FROM listings;` (a third of the prices are missing)
>
> Write your four predictions first, then check them with `q(...)`.

*Source:* PostgreSQL docs: [mathematical operators](https://www.postgresql.org/docs/current/functions-math.html), [comparison operators](https://www.postgresql.org/docs/current/functions-comparison.html), [string functions](https://www.postgresql.org/docs/current/functions-string.html)

*Your answer:*

### 1.2 · Fermi estimate

*Paper*

> Without looking at the data: how many rows of the review table (one row per listing and month with at least one review) belong to the 2,268 listings in Mitte? Write down each assumption of your estimate (how many listings have any review, for how many months on average). Then check your estimate with one query.

*Source:* [Fermi problem, Wikipedia](https://en.wikipedia.org/wiki/Fermi_problem)

*Your answer:*

### 1.3 · Count the rows of each join

*Paper*

> Table `a` has the rows (1, x), (2, y), (2, z); table `b` has (2, p), (2, q), (3, r). The first column is the key. How many rows do `a INNER JOIN b`, `a LEFT JOIN b` and `a CROSS JOIN b` return?

*Source:* original exercise for this course; background: [PostgreSQL: Joined tables](https://www.postgresql.org/docs/current/queries-table-expressions.html)

*Your answer:*

### 1.4 · Where can a `SELECT` alias be used?

*Predict*

> Each query below uses an alias from its `SELECT` list in another clause. Predict for each one whether PostgreSQL runs it or reports `column ... does not exist`, and explain your prediction with the logical order of the clauses. Then run all four.
> 1. `SELECT id, price / accommodates AS price_per_guest FROM listings WHERE price_per_guest > 50;`
> 2. `SELECT id, price / accommodates AS price_per_guest FROM listings ORDER BY price_per_guest DESC LIMIT 3;`
> 3. `SELECT id, price / accommodates AS price_per_guest FROM listings ORDER BY price_per_guest * 2 DESC LIMIT 3;`
> 4. `SELECT district AS d, COUNT(*) FROM listings GROUP BY d;`
>
> Which result does the logical order not explain? Look it up in the PostgreSQL documentation of `SELECT` (sections on `GROUP BY` and `ORDER BY`).

*Source:* [PostgreSQL: SELECT](https://www.postgresql.org/docs/current/sql-select.html)

*Your answer:*

### 1.5 · First queries on the listings

*Code*

> Answer each question with one query.
> 1. How many listings require a minimum stay of 28 nights or more?
> 2. A student looks for a cheap room in Pankow: a private room under €50 a night, a minimum stay below 28 nights, and at least 10 reviews. List these listings with id, neighbourhood, price, rating and number of reviews, best rated first. Does any price look suspicious?
> 3. How many listings in Mitte have never been reviewed according to `first_review IS NULL`? Is that the same number as with `number_of_reviews = 0`?

*Source:* original exercise for this course (Airbnb case study)

### 1.6 · Create and fill a table

*Code*

> Create a schema of your own (`CREATE SCHEMA my_trips`) and in it a table `my_trips.trips` for your own last journeys with the columns `id` (primary key), `city` (text, required), `start_date` (a date), `nights` (integer, must be at least 1) and `price_per_night` (decimal, may be unknown). Insert four trips, one of them with an unknown price. Then try to insert a trip with an `id` that exists, one with 0 nights, one without a city and one with the start date `'2026-02-30'`, and read the error messages. Finally, list the trips longer than three nights, the most recent first. What is the total cost of all trips, and how does the trip with an unknown price affect it?
>
> When you are done, remove your schema with `DROP SCHEMA my_trips CASCADE`.

*Source:* original exercise for this course; background: [PostgreSQL: CREATE TABLE](https://www.postgresql.org/docs/current/sql-createtable.html)

### 1.7 · Airbnb detective

*Pairs*

> Follow the clues; each answer is needed for the next query. Submit the final answer on paper.
> 1. Which host has the most listings in Berlin?
> 2. In which district does that host have the most listings?
> 3. In that district, which listing (of any host) has the most reviews?
> 4. In which month did that listing receive the most reviews?
>
> *Note:* clues 1 and 2 count rows per host or per district. This needs `COUNT(*)` with `GROUP BY`, which block 2 teaches in full; the pattern is `SELECT host_id, COUNT(*) AS n FROM listings GROUP BY host_id ORDER BY n DESC`.

*Source:* original exercise for this course, modelled on the [SQL Murder Mystery, Knight Lab](https://github.com/NUKnightLab/sql-mysteries)

### 1.8 · How does the database find a row? (index and query plan)

*Stretch:* *Predict, then Code*

> We look up all listings for one night: `SELECT * FROM calendar WHERE date = DATE '2026-08-01';`
> Predict: how many of the 3.8 million rows does the database read? Then run the query with `EXPLAIN ANALYZE` in front of it, before and after `CREATE INDEX idx_calendar_date ON calendar (date);`, and compare the plans and the execution times. (`EXPLAIN ANALYZE` runs the query and returns the plan as rows of text; `print("\n".join(q("EXPLAIN ANALYZE ...").iloc[:, 0]))` prints it.)
>
> Then look at the plan of `SELECT * FROM calendar WHERE listing_id = 3176;` without creating an index. Why does PostgreSQL already use an index there?
>
> When you are done, remove the index again with `DROP INDEX idx_calendar_date;` so that the database is as before.

*Source:* [PostgreSQL: Using EXPLAIN](https://www.postgresql.org/docs/current/using-explain.html)

*Your answer:*

### 1.9 · Put the query in order

*Parsons*

> The lines below form a query that lists the five cheapest entire homes in Mitte for at least four guests that can be booked for less than 28 nights, cheapest first. Put them in the right order. Then say which line changes the result most if you leave it out, and why.
>
> ```sql
> ORDER BY price
> AND accommodates >= 4 AND minimum_nights < 28
> SELECT id, price, accommodates
> LIMIT 5
> WHERE district = 'Mitte'
> AND price IS NOT NULL
> FROM listings
> AND room_type = 'Entire home/apt'
> ```

*Source:* original exercise for this course

*Your answer:*

### 1.10 · Listings first reviewed since January 2025

*Find the bug*

> A colleague counts the listings whose first review is on or after 5 January 2025, and writes the date the way it is written in Germany:
>
> ```sql
> SELECT COUNT(*) FROM listings WHERE first_review >= '05/01/2025'
> ```
>
> The query runs and returns a plausible number, but the number is wrong. Which date does PostgreSQL read from `'05/01/2025'`, which first reviews does the query count and which does it miss? Fix the query so that it does not depend on any setting, and compare the two counts. (Hint: `SHOW DateStyle`.)

*Source:* original exercise for this course; background: [PostgreSQL: Date/time input](https://www.postgresql.org/docs/current/datatype-datetime.html#DATATYPE-DATETIME-INPUT)

*Your answer:*

## Debrief

- **Joins on paper (1.3).** Collect the three numbers from several pairs. An answer of 2 rows for the inner join is the typical mistake: the duplicated key on both sides multiplies rows. Ask what this means for a join of `listings` with 365 calendar rows each.
- **Warm-ups (1.1).** Ask why `NULL = NULL` is not true, and what `COUNT(price)` counts compared with `COUNT(*)`. Point to integer division in `1/2`: it returns a surprise again in block 2 when shares are computed.
- **Aliases (1.4).** Compare the predictions with the results. The logical order explains queries 1 and 2; queries 3 and 4 show that a dialect can add its own rules on top of it. Test SQL on the database you will deploy to.
- **Dates (1.10).** Compare the two counts. Ask which way of writing a date never depends on a setting (`DATE 'YYYY-MM-DD'`).
- **First queries (1.5).** Compare the two counts of never-reviewed listings in Mitte and ask which column is the better source and why.
- **Bridge to block 2.** The detective needed counts per host and per district (`GROUP BY`), and the worked example needed two queries for two tables. Block 2 adds aggregation, joins, common table expressions, window functions, and runs SQL from Python scripts.

## Further reading

- Codd, E. F. (1970). A relational model of data for large shared data banks. *Communications of the ACM*, 13(6), 377–387. https://doi.org/10.1145/362384.362685
- UC Berkeley Data 100, course notes Fall 2023: [SQL I](https://github.com/DS-100/fa23-course-notes/blob/main/sql_I/sql_I.qmd) and [SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (BSD 3-Clause). Curated into the sections on relational databases, keys, creating tables and queries.
- Course team: [extras/02-loading-with-constraints.ipynb](extras/02-loading-with-constraints.ipynb), schema-first loading of the whole case study.
- PostgreSQL documentation: [Tutorial: The SQL language](https://www.postgresql.org/docs/current/tutorial-sql.html), [Data types](https://www.postgresql.org/docs/current/datatype.html), [Date/time types](https://www.postgresql.org/docs/current/datatype-datetime.html), [CREATE TABLE](https://www.postgresql.org/docs/current/sql-createtable.html), [Constraints](https://www.postgresql.org/docs/current/ddl-constraints.html), [COPY](https://www.postgresql.org/docs/current/sql-copy.html), [SELECT](https://www.postgresql.org/docs/current/sql-select.html), [Pattern matching](https://www.postgresql.org/docs/current/functions-matching.html), [Using EXPLAIN](https://www.postgresql.org/docs/current/using-explain.html), [History](https://www.postgresql.org/docs/current/history.html).
- psycopg 3 documentation: [Basic module usage](https://www.psycopg.org/psycopg3/docs/basic/usage.html), [Using COPY](https://www.psycopg.org/psycopg3/docs/basic/copy.html).
- [pixeltable-pgserver](https://github.com/pixeltable/pixeltable-pgserver): the package that starts the local PostgreSQL server.